# 🔬 **MorphoBricks | Preprocessing**
___

This notebook prepares an experiment for the other MorphoBricks bricks. It measures the
brightness of every 2D plane of your images (TIFF or OME-Zarr) and saves one **intensity histogram per
channel** of every plane in a table, **plane_histograms.csv**: one row per plane, the channels side by
side. Later bricks use this table to normalise images per
condition, repeat or image without reading the pixels again.

### What the notebook does

- **Checks your experiment folder:** the folder names become the labels of your images
  (condition, biological repeat, technical repeat).
- **Asks you about the experiment design:** which biological repeats belong together across
  conditions, whether they are comparable, and which conditions are controls.
- **Asks you to confirm how the images are organised:** which dimension is depth, channel or
  time. It shows a preview so you can check. Then it asks whether they are fluorescence or RGB
  colour images.
- **Computes the histograms** and writes them, with a record of what was done, to a separate
  output folder.
- **Checks the intensities** of every image from the histograms: images far from the others of
  their biological repeat, shifted biological repeats, saturated images and images without
  spread, with plots per channel.
- **Shows pooled statistics** per condition, repeat or image.

### Quick workflow

1. Run the **Step 0** cell to start.
2. **Step 1:** paste or **Browse…** to your experiment folder, press **Check folder**, read the
   messages and, if the labels are right, press **Yes, the labels are correct**. Then confirm the
   biological repeats with **Yes, they are comparable**.
3. **Step 2:** check the role of each axis, press **Show preview**, press
   **Yes, the preview looks right** and then **Confirm axis roles**. Check the suggested image
   type and press **Yes, this is the right image type**.
4. **Step 3:** press **Compute histograms**. This writes the table the next bricks need. Read
   the **Intensity checks** that follow.
5. **Step 4 (optional):** press **Calculate statistics** to compare your groups and check the
   intensities of any table.

### Before you start

- [ ] Your images are **.tif** or **.tiff** files, or OME-Zarr folders (**.ome.zarr**), all with the same pixel type.
- [ ] They are organised in folders like the example below (up to three levels).
- [ ] You know roughly how many slices, channels and time points they have.

> 🛡️ **Your images are never changed.** The results go to a separate folder next to the
> experiment folder (or one you choose), never inside it.

```text
📂 experiment_2026_10                ← select this folder
├── 📂 control                        condition
│   ├── 📂 bio_rep1                   biological repeat
│   │   ├── 📂 tech_rep1              technical repeat
│   │   │   ├── 📄 image_01.tif
│   │   │   └── 📄 image_02.tif
│   │   └── 📂 tech_rep2
│   │       └── 📄 image_01.tif
└── 📂 treated
    └── 📂 bio_rep1                   (no technical repeat folder → "single")
        └── 📄 image_01.tif
```

___
## 🧰 **Step 0: Start the notebook**

Click the cell below and press **Shift + Enter** (or the ▶ button in the toolbar). A green box
appears when everything is ready.

> 💡 **Tip:** the code of each cell is hidden; you only need to run the cells, from top to bottom,
> one after the other.

In [ ]:
# @markdown ### Run to start the preprocessing notebook
import subprocess
import sys
from importlib import import_module, invalidate_caches
from importlib.util import find_spec
from pathlib import Path

# Current version of the notebook the user is running
current_version = "0.0.1"
notebook_name = "MorphoBricks_Preprocessing"


def _start_preprocessing_notebook() -> object:
    """Prepare this kernel's environment and return the notebook interface."""
    # This standalone cell can run before the package's Python requirement is checked.
    if sys.version_info < (3, 11):  # noqa: UP036
        raise RuntimeError("Select a Python 3.11 or newer notebook kernel, then run Step 0 again.")

    environment = "Google Colab" if "google.colab" in sys.modules else "local Jupyter"
    print(f"Checking the {environment} environment...", flush=True)
    current = Path.cwd().resolve()
    checkout = next(
        (
            folder
            for folder in (current, *current.parents)
            if (folder / "pyproject.toml").is_file()
            and (folder / "src/morphobricks_preprocessing/__init__.py").is_file()
        ),
        None,
    )
    if checkout is not None:
        source = checkout / "src"
        loaded = sys.modules.get("morphobricks_preprocessing")
        if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(source):
            raise RuntimeError(
                "This kernel has already loaded another copy of MorphoBricks preprocessing. "
                "Restart the kernel, then run Step 0 to use this checkout."
            )
        # A local source tree works immediately, including in a git worktree.
        sys.path[:] = [str(source), *(entry for entry in sys.path if entry != str(source))]
        invalidate_caches()
        install_target = ["-e", f"{checkout}[notebook]"]
        origin = f"the local checkout at {checkout}"
    elif find_spec("morphobricks_preprocessing") is not None:
        install_target = ["morphobricks-preprocessing[notebook]"]
        origin = "the installed package"
    else:
        install_target = [
            "morphobricks-preprocessing[notebook] @ "
            "git+https://github.com/CellMigrationLab/morphobricks-preprocessing.git"
        ]
        origin = "the GitHub installation"

    required = (
        "morphobricks_preprocessing",
        "morphobricks_normalization",
        "numpy",
        "tifffile",
        "ipywidgets",
        "matplotlib",
    )
    missing = [name for name in required if find_spec(name) is None]
    if missing:
        print(f"Installing notebook requirements ({', '.join(missing)})...", flush=True)
        try:
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "--disable-pip-version-check",
                    *install_target,
                ],
                check=True,
            )
        except (subprocess.CalledProcessError, OSError) as exc:
            raise RuntimeError(
                "Notebook setup could not install its requirements. Check the installation "
                "output above, internet access and write permissions, then rerun Step 0. "
                "Installing from GitHub also needs Git."
            ) from exc
        invalidate_caches()
        missing = [name for name in required if find_spec(name) is None]
        if missing:
            raise RuntimeError(
                f"Installation finished, but this kernel still cannot find {', '.join(missing)}. "
                "Restart the kernel and run Step 0 again."
            )

    print(f"Using {origin}.\nNotebook Python: {sys.executable}", flush=True)
    try:
        interface = import_module("morphobricks_preprocessing.notebook_ui")
    except ImportError as exc:
        raise RuntimeError(
            "The package was found, but the notebook interface could not be loaded. "
            "Read the original error below. If packages were changed in this session, "
            "restart the kernel and run Step 0 again."
        ) from exc
    return interface.PreprocessingNotebook()


notebook = _start_preprocessing_notebook()
notebook.welcome()

___
## 📂 **Step 1: Check the experiment folder**

Run the cell below, then:

1. Paste the path of your experiment folder into **Experiment folder**, or click **Browse…** to
   choose it. The **Output folder** box shows where the results will go; leave it empty to use the
   default.
2. Click **Check folder**. Only folder names and image headers are read, so this is quick.
3. Read the summary tiles and the **Messages**. Red messages must be fixed before you can continue;
   amber ones are for information.
4. Check the **Labels from the folder names** table. If every folder has the right labels,
   press the green **Yes, the labels are correct** button in the yellow box under the table.
   It turns green when confirmed; press **Undo** if you change your mind. Step 3 only runs once
   the labels are confirmed.
5. Answer the questions under **Experiment design**, which appear once the labels are confirmed:
   - If a biological repeat has a different folder name in some condition, choose its
     **Repeat group** in the dropdown, so the same repeat is paired across conditions.
   - For each repeat group, if its images were prepared and acquired under the same
     circumstances, press **Yes, they are comparable**. Step 3 only runs once every group is
     confirmed.
   - Optionally, tick the control conditions.

   The answers are saved in **experiment_design.csv** in the output folder. Next time, with the
   same folders, they are loaded and not asked again; press **Change the experiment design** to
   look at them.
6. Look at **Dataset consistency**. Its table counts the images of every condition and
   biological repeat; amber cells have far fewer or far more images than the others, or a
   biological repeat that a condition does not have. Below it, **Show consistency warnings**
   lists differences between the images: pixel type, dimensions, sizes, numbers of channels,
   slices or time points, channel names and pixel sizes. Warnings never stop you; each says
   what to check. Different numbers of channels, or different channel names at the same channel
   number, are errors: every image needs the same channels in the same order.

> 💡 **Tip:** to change a label, rename the folder and click **Check folder** again.
>
> 🩹 **A damaged image?** Click **Exclude these images and check again** to leave it out, or list
> images under **Exclude images (optional)**, one per line.

In [ ]:
# @markdown ### Run to show the folder check
notebook.folder_step()

___
## 🧭 **Step 2: Review the axis roles**

A TIFF or OME-Zarr image is a block of numbers with several dimensions. This step tells the brick
what each dimension means, so that every image can be cut into flat 2D planes:

| Role | Meaning | Example |
|---|---|---|
| **X** | width of the image (columns) | 1024 pixels |
| **Y** | height of the image (rows) | 1024 pixels |
| **Z** | depth: focal planes of a z-stack | 25 slices |
| **C** | channel: stains or colours | DAPI, GFP, RFP |
| **T** | time: frames of a time-lapse | 120 time points |

Run the cell below, then:

1. Check the suggested role in each box. The line under each box says what the choice means.
2. Click **Show preview** and move the sliders. Each picture should look like a normal image of your
   sample. Stripes or a smeared picture mean X or Y is on the wrong dimension.
3. If the preview looks right, press **Yes, the preview looks right** in the yellow box under the
   picture. Changing a role afterwards asks you to check the preview again.
4. Press **Confirm axis roles**. The roles are locked until you press **Change axis roles**.
5. Under **Image type**, check the suggestion: **Fluorescence** (each channel is a separate stain
   or marker) or **RGB** (colour images, such as brightfield or histology stains). RGB is suggested
   when the images are stored as RGB colour. If the suggestion is right, press
   **Yes, this is the right image type**; otherwise choose the other type first. Later bricks
   normalise the two types differently; the histograms are the same. Step 3 only runs once the
   type is confirmed.

> 💡 **Tip:** if some images lack a dimension the others have (often a single channel), a yellow box
> lists them. Press **Yes, they have a single plane there** if that is the case.
>
> ℹ️ RGB can only be chosen for images with exactly three channels of a whole-number pixel type
> (8- or 16-bit colour); otherwise the card says why.

In [ ]:
# @markdown ### Run to review the axis roles
notebook.axis_step()

___
## 🚀 **Step 3: Compute the histograms**

Run the cell below. The **Before you run** box lists anything still to do. **What will happen** says
which folder and files will be written, whether an earlier run was found, and which images
are too large for the memory limit and will be read in blocks. Then click
**Compute histograms**.

> 💡 **Running again later** only reads images that are new or changed. Rows of deleted or excluded
> images are removed. The recommended settings suit most experiments; untick
> **Use the recommended settings** to change the number of bins, the memory limit, or to recompute
> everything.
>
> 🗂️ **(Optional) Image conversion:** by default the images are read directly and nothing extra is
> saved. You can instead **convert the TIFF images to OME-Zarr** first; the histograms are then computed
> from the converted copies, which are stored in the **ome_zarr** folder of the output. ⚠️ This stores
> the images a second time, so the space they occupy is duplicated; the card shows how much space is
> needed and lets you choose the chunk size, compression and pyramid levels. It needs the zarr package.
>
> 🛡️ Nothing is written unless the whole run succeeds.

After the run, **Intensity checks** compares the images using the table just written (no image is
read again, nothing more is written):

- warnings about images far from the other images of their condition and biological repeat,
  biological repeats shifted from the other repeats of their condition, saturated images and
  images without intensity spread, each with the channel (**C0**, **C1**, …), the image or
  repeat, and the values compared, and what to do;
- box plots of the image medians and high percentiles per condition and biological repeat
  (red diamonds: images with a warning; amber boxes: shifted repeats), and the intensity
  distribution of each biological repeat within each condition. Choose the channel above the
  plots.

> ℹ️ Images are compared only within their condition, so a treatment effect is never reported.
> The limits (3 robust deviations, 1 % saturated pixels, at least 5 images) are under
> **Check settings**; changing them updates the warnings at once, and the next run records them.

In [ ]:
# @markdown ### Run to show the settings and the run button
notebook.run_step()

___
## 📊 **Step 4: Calculate the statistics (optional)**

> ℹ️ **This step is optional.** The histogram table written in step 3 is all that the next
> MorphoBricks bricks need. This step only reads it, to let you compare groups and check that the
> brightness looks as expected; it writes no files.

Run the cell below, then:

1. **Histogram table** is filled in with the table from step 3. To look at another experiment,
   paste the path of its plane_histograms.csv or press **Browse…**.
2. Choose how to group the planes (**Condition**, **Biological repeat**, **Technical repeat** or
   **Image**) and the percentiles to report.
3. Press **Calculate statistics**. The table lists pooled statistics per group, and the plot
   compares their intensity distributions. **Intensity checks** shows the same warnings and
   plots as step 3 for the chosen table, with its own **Check settings**.

In [ ]:
# @markdown ### Run to show the statistics
notebook.results_step()

___
## ✅ **After running**

- [ ] The output folder contains **plane_histograms.csv**, **experiment_design.csv** and
  **run_manifest.json**.
- [ ] Keep these files together; the next MorphoBricks brick reads them.
- [ ] Rerun this notebook whenever you add, change or remove images.
- [ ] If you chose to convert the TIFF images, the output folder also has an **ome_zarr** folder
  with one OME-Zarr copy per TIFF image.

### Common problems

**"The images use different pixel types"**: all images of an experiment must have the same type,
for example 16-bit. Process them separately or convert them.

**"… not confirmed yet"**: answer the yellow questions under **Experiment design** in step 1
with **Yes, they are comparable**. If some images are not comparable, process them as a
separate experiment.

**"A repeat group is one biological repeat across the conditions, so it can hold only one folder
of each condition"**: two folders of the same condition were put in one repeat group; move one of
them into another group.

**"No role is chosen for the axis"**: a dimension with several planes needs Z, C or T in step 2.

**"The images have different numbers of channels"**: each row of the table holds every channel of
a plane, so all images need the same channels. Process the others as a separate experiment, or
exclude them; if a dimension got the role C by mistake, correct it in step 2.

**"… cannot be read within the memory limit, not even in blocks"**: untick *Use the
recommended settings* in step 3 and raise the memory limit, if the computer has enough
memory, or save the images again as tiled TIFF files or OME-Zarr. Planes that are only too
large to read whole are read in blocks automatically, with the same results.

**"The number of images differs strongly between conditions and biological repeats"**: some
folders have far fewer or far more images than the others. Nothing to do if this is expected;
otherwise check for missing, misplaced or duplicated images.

**"The physical pixel sizes in the image metadata differ"**: the images may come from different
objectives. Nothing to do if that is intended; otherwise check the microscope settings.

**"RGB images need exactly three colour channels of a whole-number pixel type"**: choose
**Fluorescence** in step 2, or check the axis roles: the RGB samples must have the role C.

**"Image …, channel C0: its median intensity … is … robust deviations from …"**: an
intensity check, never an error. Look at the image: it may be out of focus, contain debris or
have been acquired with other settings. If it is faulty, exclude it and run again.

**The preview looks like stripes**: X or Y is on the wrong dimension; change the roles in step 2.

More help: the [user guide](../docs/user_guide.md).